# 00 Python 核心语法与数据模型

> 覆盖范围：对象模型（可变/不可变、深浅拷贝、`is` vs `==`、可变默认参数陷阱）、容器与推导式、函数式编程（lambda/闭包/装饰器的**原理**）、迭代器与生成器（yield 挂起恢复原理）、类与魔术方法、GIL 与并发选型、性能优化。
> 使用方式：每节先读**原理讲解**（为什么存在 + 底层机制），再跑**代码**验证，用**对比表**速记，最后自测**面试追问**。代码均为 Python 3 可直接运行。

---

## 1. 对象模型：一切皆对象

### 1.1 什么是「数据模型」

Python 的设计哲学是**一切皆对象**：数字、字符串、函数、类、模块——全部都是对象。每个对象都有三个属性：

1. **身份（identity）**：`id(obj)` 返回，可理解为对象在内存中的地址（CPython 中即 `PyObject*` 指针）；
2. **类型（type）**：`type(obj)` 返回，决定对象支持哪些操作；
3. **值（value）**：对象携带的数据。

**关键推论**：变量名只是「**名字 → 对象**」的绑定（实际上是指向 `PyObject` 的指针），不是存放值的盒子。`a = 1; b = a` 后 `a`、`b` **指向同一个 int 对象**。

### 1.2 不可变对象为什么存在

`str` / `int` / `tuple` 等不可变类型不是拍脑袋设计的，有三个实际收益：

- **可作为哈希键**：dict 的键 / set 的元素要求「哈希值在生命周期内不变」——可变对象哈希值变了会破坏哈希表结构，所以 list/dict/set 不可哈希；
- **线程安全、可安全共享**：不可变对象被多处引用不会意外修改；
- **允许驻留（intern）与缓存**：小整数、短字符串可复用同一对象，节省内存（后面 `is` 小节会用到）。

| 类别 | 类型 | 说明 |
| --- | --- | --- |
| 不可变 | `int` / `float` / `bool` / `complex` / `str` / `tuple` / `frozenset` / `bytes` | 哈希可用，可作 dict 键 / set 元素 |
| 可变 | `list` / `dict` / `set` / `bytearray` / 自定义类实例 | 哈希不可用（除非显式定义 `__hash__`） |

> ⚠️ **最易错**：`tuple` 不可变指的是**元组本身**不能增删改元素，但它内部的**元素若是可变对象**（如 list），仍可被修改——「元组不可变」≠「元组内容不可变」。

### 1.3 `+=` 的隐藏语义（原地 or 新建）

`a += x` 到底发生什么，取决于 `a` 的类型：

- 不可变类型（str/int/tuple）：`a += x` 等价于 `a = a + x`，**创建新对象**再让名字绑定过去，`id(a)` 变化；
- 可变类型（list）：调用 `__iadd__`，**原地扩展**，`id(a)` 不变；
- 而对 `tuple` 来说 `+=` 会**报错**（元素不可变）——除了「装着可变元素的 tuple」这种边角情况。

### 1.4 深浅拷贝（面试高频）

拷贝的本质是「制造新对象」，但**复制多深**是个问题：

- `b = a`：**不拷贝**，只是新增一个名字指向同一对象（赋值永远不拷贝）；
- `copy.copy(a)`：**浅拷贝**——新建顶层容器，内部元素仍共享引用；
- `copy.deepcopy(a)`：**深拷贝**——递归复制所有层，生成完全独立的副本；
- 深拷贝还支持**循环引用**（a 引用 b、b 引用 a 也能安全复制，内部用 memo 记录已拷贝对象）；
- 自定义类可覆写 `__copy__` / `__deepcopy__` 控制拷贝行为（例如只复制某些字段）。

> **怎么选**：纯数据（数字/str）多用浅拷贝即可（元素不可变，共享无害）；嵌套可变容器要独立修改时用深拷贝；大结构慎用深拷贝（性能/内存翻倍）。

### 1.5 `is` vs `==`：身份 vs 值

- `==` 比较**值**（调用左操作数的 `__eq__`）；
- `is` 比较**身份**（`id()` 是否相同）。

「小整数缓存」细节：CPython 对 `-5 ~ 256` 的整数**在启动时就预先创建**，所以 `a = 256; b = 256; a is b` 为 `True`；而 `257` 每次新建对象，`is` 为 `False`。短字符串（不含空格、类标识符风格）也会驻留。

> ⚠️ **铁律**：`is` 只用于「与 `None` 比较」或「确认是否同一对象」；**永远不要**用 `is` 判断数值/字符串内容相等——那是 `==` 的职责，`is` 的结果依赖驻留缓存，不可靠。


## 核心概念

### (a) 🗂️ Python 从「对象模型」到「并发选型」的主线

这篇把 Python 的底层机制串成一条主线：一切皆对象的对象模型、容器与推导式、函数式编程、迭代器与生成器、类与魔术方法，最后落到 GIL 与并发选型和性能优化——理解了这条线，面试和写代码都不再靠背。

```mermaid
flowchart TB
    Node1["🧠 Python 一切皆对象"] --> Node2["📦 可变 vs 不可变"]
    Node1 --> Node3["🧰 容器与推导式"]
    Node2 --> Node4["🔍 is vs == 与深浅拷贝"]
    Node3 --> Node5["🔁 迭代器与生成器"]
    Node3 --> Node6["🧩 函数式：lambda / 闭包 / 装饰器"]
    Node5 --> Node7["🏷️ 类与魔术方法"]
    Node6 --> Node7
    Node7 --> Node8["🔒 GIL 与并发选型"]
    Node8 --> Node9["🚀 性能优化速记"]
```

- **🧠 Python 一切皆对象**：数字、字符串、函数、类、模块全都是对象，变量名只是指向对象的「名字标签」，不是装值的盒子。搞懂这一点，后面可变/不可变、拷贝、`is` 全都顺理成章。
- **📦 可变 vs 不可变**：`str`/`int`/`tuple` 不可变，能哈希、可安全共享、可驻留缓存；`list`/`dict`/`set` 可变，可原地修改。注意「元组不可变」只是结构不能增删，里面的 list 照样能改。
- **🧰 容器与推导式**：选容器就是选底层数据结构——`list` 是动态数组（尾部 O(1)）、`dict`/`set` 是哈希表（查找平均 O(1)）、`deque` 两端 O(1)。推导式是「一行建容器」的语法糖，比 for+append 快一点且不泄漏循环变量。
- **🔍 is vs == 与深浅拷贝**：`==` 比内容（调 `__eq__`），`is` 比身份（`id()`）。赋值 `b = a` 不拷贝，`copy.copy` 只复制顶层、内层仍共享，`deepcopy` 递归复制全部；小整数驻留让 `256 is 256` 为 True 而 `257 is 257` 为 False，所以别用 `is` 比数值。
- **🔁 迭代器与生成器**：`for` 的底层是 `iter()` 拿迭代器、反复 `__next__()`、捕获 `StopIteration` 结束。函数里写 `yield` 就成了生成器：执行到 `yield` 挂起保存现场，下次 `next()` 恢复——内存 O(1)，还能串成流水线处理无限序列。
- **🧩 函数式：lambda / 闭包 / 装饰器**：lambda 是单表达式匿名函数；闭包把外层变量封存在 `cell` 里供内层函数用（注意延迟绑定陷阱和 `nonlocal`）；装饰器 `@deco` 本质就是 `f = deco(f)`，记得用 `functools.wraps` 保住元信息。
- **🏷️ 类与魔术方法**：构造走 `__new__` → `__init__`；`__repr__`/`__eq__`/`__len__`/`__call__` 等魔术方法是协议钩子，让对象融入 Python 语法。多继承靠 MRO（C3 线性化）保证每个基类只出现一次，`super()` 沿 MRO 向后传播。
- **🔒 GIL 与并发选型**：GIL 是 CPython 为保护引用计数加的全局锁，同一时刻只有一个线程执行字节码，所以多线程并行不了纯 Python 的 CPU 计算。I/O 密集用 `threading`/`asyncio`，CPU 密集用 `multiprocessing`；asyncio 适合海量轻量 I/O。
- **🚀 性能优化速记**：内置函数优先、推导式胜过 for+append、局部变量快于全局变量、`join` 拼字符串、用 `set`/`dict` 查重 O(1)、`__slots__` 省内存。但先 `timeit`/`cProfile` 定位瓶颈再动手，别凭感觉优化。


In [ ]:
# ---- matplotlib 中文字体（防图内中文乱码）----
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Songti SC', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
# -------------------------------------------
import copy

# 绑定 vs 浅拷贝 vs 深拷贝
a = [[1, 2], [3, 4]]
b = a                      # 绑定：同一个对象
c = copy.copy(a)           # 浅拷贝：顶层新列表，内部子列表共享
d = copy.deepcopy(a)       # 深拷贝：全部独立

a[0][0] = 99
print('a =', a)
print('b =', b, ' <- 绑定，随 a 改变')
print('c =', c, ' <- 浅拷贝，子列表随 a 改变')
print('d =', d, ' <- 深拷贝，完全独立')

# 不可变对象的 += 产生新对象
s = 'ab'
print('str id 前:', id(s))
s += 'c'
print('str id 后:', id(s), ' <- 新对象')

lst = [1, 2]
print('list id 前:', id(lst))
lst += [3]
print('list id 后:', id(lst), ' <- 原地修改，id 不变')

In [ ]:
# is vs ==：驻留缓存演示 + 可变默认参数陷阱

# 小整数驻留（-5 ~ 256 启动时预创建）
a, b = 256, 256
c, d = int('257'), int('257')   # 用 int() 强制新建对象，避免同一行赋值复用常量池
print('256 is 256:', a is b, ' <- 驻留范围内，同一对象')
print('257 is 257:', c is d, ' <- 超出驻留范围，int() 每次新建对象')
print('None 比较习惯:', a is None, ' <- is 用于 None 是规范用法')


# 可变默认参数陷阱：默认参数在函数定义时只求值一次！
def bad_append(item, lst=[]):
    lst.append(item)
    return lst

print('第一次:', bad_append(1))
print('第二次:', bad_append(2), ' <- 同一个默认 list 被共享，累积了！')

def ok_append(item, lst=None):
    lst = [] if lst is None else lst
    lst.append(item)
    return lst

print('修复后:', ok_append(1), ok_append(2), ' <- 每次新建，互不影响')

## 2. 容器与推导式

### 2.1 容器选型：底层结构决定复杂度

选容器的本质是选**底层数据结构**：

| 容器 | 底层 | 特点 |
| --- | --- | --- |
| `list` | 动态数组（PyObject* 数组） | 尾部 O(1)，中间插入/删除 O(n)，按索引随机访问 O(1)，有序可重复 |
| `tuple` | 固定数组 | 不可变、可哈希、可作键，拆包方便，比 list 省内存 |
| `dict` | 哈希表（开放寻址） | 键唯一，查找/插入平均 O(1)，Python 3.7+ 保持插入顺序 |
| `set` | 哈希表 | 去重、成员判断 O(1)，无序 |
| `collections.deque` | 双向链表/块数组 | 两端 O(1) 插入/弹出，中间 O(n) |
| `collections.Counter` | dict 子类 | 频次统计，`most_common` 排序 |
| `collections.namedtuple` | tuple 子类 | 具名字段的小数据结构 |

**list 扩容机制**：列表初始容量约 8，满了按约 1.125 倍（实际 `newsize = size + size>>3 + ...`）扩容并搬运，均摊后 append 仍 O(1)。

**dict 的哈希细节**：键先算 `hash(key)`，再对表长取模定位；发生冲突（哈希碰撞）时用**开放寻址**探查下一个空位。因此要求键可哈希且哈希稳定——这又回到 1.1 节「不可变才能哈希」。

### 2.2 推导式（Comprehension）

推导式是「一行建容器」的语法糖，底层比等价 for 循环快（少一次属性查找与 append 调用）：

- 列表推导 `[expr for x in it if cond]`；
- 字典推导 `{k: v for ...}`；
- 集合推导 `{x for ...}`；
- 生成器表达式 `(expr for ...)`（惰性，见 §4）；
- 支持**多层嵌套与双 if**：`[ (x, y) for x in range(3) for y in range(3) if x != y ]`，层叠顺序与普通 for 嵌套一致（先写外层）。

> ⚠️ **易错点**：① 推导式有**独立作用域**（不会泄漏循环变量到外层）；② 三层以上嵌套时**可读性崩塌**，改用普通循环 + 注释；③ 别把 `sorted()` 与「推导式 + 排序」混为一谈。


In [ ]:
from collections import deque, Counter, namedtuple, defaultdict

# 推导式
squares = [x * x for x in range(10)]
evens = [x for x in range(10) if x % 2 == 0]
sq_dict = {x: x * x for x in range(5)}
sq_set = {x % 3 for x in range(10)}
pairs = [(x, y) for x in range(3) for y in range(3) if x != y]
print('squares:', squares)
print('evens  :', evens)
print('dict   :', sq_dict)
print('set    :', sq_set)
print('嵌套去对角:', pairs[:6], '... 共', len(pairs), '个')

# deque / Counter / namedtuple / defaultdict
d = deque([1, 2, 3])
d.appendleft(0); d.append(4)
print('deque  :', list(d))

c = Counter('abracadabra')
print('Counter:', c.most_common(3))

Pt = namedtuple('Pt', ['x', 'y'])
p = Pt(1, 2)
print('namedtuple:', p, p.x, p.y, '| 可解包:', tuple(p))

dd = defaultdict(list)
for ch in 'abac':
    dd[ch].append(1)
print('defaultdict:', dict(dd))

## 3. 函数式编程：lambda / 闭包 / 装饰器

### 3.1 lambda：匿名单表达式函数

`lambda x: 表达式` 等价于 `def f(x): return 表达式`，但**只能写单表达式**（不能有赋值/多语句）。它常用于「一次性小函数」，配合：

- `sorted(iterable, key=...)` / `list.sort(key=...)`——按自定义键排序；
- `map(func, it)` / `filter(func, it)`（Python 3 返回惰性迭代器）；
- `functools.reduce` / `functools.partial`。

> 面试常考：**`map`/`filter` vs 推导式**——推导式通常更清晰，面试推荐两者都能写；`reduce` 能不用就不用（可读性差）。

### 3.2 闭包：函数 + 捕获的自由变量

**闭包（closure）= 函数 + 它引用的外层自由变量的绑定**。当内层函数引用了外层函数的变量时，该变量会「被封存」进内层函数的 `__closure__`，即使外层函数已返回，内层仍能读写它。

底层机制：外层局部变量存在**函数帧（frame）**里，内层通过 `cell` 对象引用它们，所以闭包里的变量是**共享同一份**（不是拷贝）。

> ⚠️ **两个经典陷阱**：
> 1. **延迟绑定**：循环创建闭包时，若直接引用循环变量 `i`，所有闭包共享的是**同一个 cell**，循环结束后 `i` 停在终值——所以 `[lambda: i for i in range(3)]` 三个函数都返回 2。修复：用默认参数 `lambda i=i: i` 把当前值「钉死」到函数上（默认参数在定义时求值）。
> 2. **修改外层变量要 `nonlocal`**：只读外层变量可以直接引用；但在内层**赋值**会把它变成内层局部变量——需要 `nonlocal` 声明才能修改外层 cell。

### 3.3 装饰器：函数替换的语法糖

`@deco` 写在函数定义上，等价于 `f = deco(f)`——**装饰器是一个接收函数、返回新函数的可调用对象**。原理三步：

1. 定义普通函数 `f`；
2. 调用 `deco(f)` 得到包装后的函数（通常是闭包 `wrapper`）；
3. 把名字 `f` 重新绑定到包装函数上。

**为什么 `functools.wraps`**：不包的话，`f.__name__`、`f.__doc__`、`f.__dict__` 全变成 wrapper 的，调试与内省（如 Sphinx 文档）全乱。`@wraps(f)` 用 `functools.update_wrapper` 把元信息拷贝回来。

**进阶形态**：

- **带参数装饰器**：`@deco(arg)` 实际是「先调用 `deco(arg)` 返回真正的装饰器，再装饰」——多包一层；
- **多个装饰器**：`@A @B def f` 按**从下往上**依次作用：`f = A(B(f))`；
- **类实现装饰器**：实现 `__call__` 的类实例可作装饰器，适合需要携带状态的场景；
- 内置装饰器：`@property`（属性化 + setter 校验）、`@staticmethod`（无 self/cls）、`@classmethod`（第一参数是 cls，支持子类改写）、`@functools.lru_cache`（记忆化）。


In [ ]:
# lambda 与高阶函数
pairs = [(1, 'b'), (3, 'a'), (2, 'c')]
pairs.sort(key=lambda p: p[1])
print('按第二元素排序:', pairs)

print('map :', list(map(lambda x: x * 2, [1, 2, 3])))
print('filter:', list(filter(lambda x: x > 2, [1, 2, 3, 4])))

# 闭包：计数器
def make_counter():
    n = 0
    def inc():
        nonlocal n        # 修改外层变量必须 nonlocal
        n += 1
        return n
    return inc

c1 = make_counter()
print('闭包计数:', c1(), c1(), c1(), '| 闭包 cell:', make_counter().__closure__ is not None)

# 延迟绑定陷阱
funcs = [lambda: i for i in range(3)]
print('延迟绑定陷阱:', [f() for f in funcs])   # 全是 2

funcs2 = [lambda i=i: i for i in range(3)]     # 默认参数立即绑定
print('修复后:', [f() for f in funcs2])        # 0 1 2

# 装饰器：计时
import functools, time

def timed(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        r = fn(*args, **kwargs)
        print(f'{fn.__name__} 耗时 {time.perf_counter()-t0:.6f}s')
        return r
    return wrapper

@timed
def slow_sum(n):
    """求 0..n-1 的和"""
    return sum(range(n))

print('装饰器结果:', slow_sum(10**6), '| 元信息保留:', slow_sum.__name__)

# 多个装饰器方向演示：从下往上
def deco_a(f):
    @functools.wraps(f)
    def w(): return 'A(' + f() + ')'
    return w

def deco_b(f):
    @functools.wraps(f)
    def w(): return 'B(' + f() + ')'
    return w

@deco_a
@deco_b
def inner(): return 'f'
print('多层装饰器方向:', inner(), ' <- 等价 A(B(f))')

## 4. 迭代器与生成器

### 4.1 迭代协议（Iterator Protocol）

`for x in obj` 的完整底层流程：

1. 调用 `iter(obj)` → 得到迭代器（若 obj 实现了 `__iter__` 返回迭代器，或用 `__getitem__` 兜底）；
2. 反复调用迭代器的 `__next__()`；
3. 捕获 `StopIteration` 异常时循环结束。

所以「可迭代对象（iterable）」只需实现 `__iter__`（或 `__getitem__`），「迭代器（iterator）」还必须实现 `__next__` 且 `__iter__` 返回自身。**迭代器是一次性的**：耗尽后再次迭代为空。

### 4.2 生成器：惰性求值的核心

生成器 = 函数体内含 `yield`，调用它**不执行函数体**，而是返回一个生成器对象（既是迭代器又可迭代）。执行过程是**挂起/恢复**：

- 首次 `next(g)` 执行到第一个 `yield`，把值产出并**挂起**（保存整个帧：局部变量、指令指针）；
- 再次 `next(g)` 从挂起处**恢复**继续执行；
- `StopIteration` 自然结束。

**收益**：

- **内存 O(1)**：处理超大序列不需要全部装入内存（对比 `list(range(10**7))` 占几百 MB，`range`/生成器只保存状态）；
- **流水线**：多个生成器串联 `(f(x) for x in gen)` 逐元素流动，不产生中间大列表；
- **无限序列**：斐波那契、素数筛可以「无限产出、按需取前 N 个」。

**进阶**：

- `yield from subgen`：把子生成器的产出**透传**（委托迭代），并正确处理返回值；
- `g.send(value)`：向生成器**注入值**，`value` 成为当前 `yield` 表达式的求值结果（协程的雏形）；
- `g.close()` / 扔 `GeneratorExit`：外部关闭生成器，触发 finally；
- **生成器表达式** `(x*x for x in ...)` 是匿名生成器的语法糖。

> **面试追问**：生成器 vs 列表推导哪个快？——小数据量列表推导更快（生成器有迭代器开销）；大数据量/流式处理生成器省内存，是「正确」答案。


In [ ]:
# 手写迭代器：实现 __iter__ / __next__
class FibIter:
    def __init__(self, n):
        self.n, self.i, self.a, self.b = n, 0, 0, 1
    def __iter__(self):
        return self
    def __next__(self):
        if self.i >= self.n:
            raise StopIteration
        self.i += 1
        self.a, self.b = self.b, self.a + self.b
        return self.a

print('手写迭代器:', list(FibIter(10)))

# 生成器：yield 惰性
def fib_gen(n):
    a, b = 0, 1
    for _ in range(n):
        yield a
        a, b = b, a + b

print('生成器     :', list(fib_gen(10)))

# 惰性 vs 一次性
g = (x * x for x in range(5))
print('生成器表达式:', list(g))
print('再次迭代为空:', list(g), ' <- 一次消费')

# 内存对比（概念演示：range 不建列表）
import sys
r = range(10**7)
print('range 对象内存(字节):', sys.getsizeof(r), ' <- 惰性，不存元素')

# yield from 委托 + send 注入
def chain(*gens):
    for g in gens:
        yield from g

print('yield from:', list(chain(range(3), range(3, 5))))

def echo():
    while True:
        got = yield 'ready'
        if got is None:
            continue
        yield f'echo:{got}'

e = echo()
print('send 演示:', next(e), '->', e.send('hi'), '->', e.send('bye'))

## 5. 类与魔术方法

### 5.1 对象的生命周期与构造

创建对象时：`__new__`（分配内存、返回实例）→ `__init__`（初始化属性）。`__new__` 是静态方法，常规用法是让 `object.__new__(cls)` 干活；**不可变类型 / 单例 / 元类**才需要覆写它。

### 5.2 常用魔术方法一览（协议钩子）

| 方法 | 触发 | 用途/注意 |
| --- | --- | --- |
| `__init__` | 构造 | 初始化实例（先 `__new__` 后 `__init__`） |
| `__repr__` / `__str__` | `repr()` / `str()` / `print` | repr 面向开发者（唯一、可重建）；str 面向用户；`__repr__` 缺失时提供默认 |
| `__eq__` / `__hash__` | `==` / `hash()` | 定义了 `__eq__` 后 CPython 自动把 `__hash__` 置 `None`（不可哈希）——**需显式重定义** |
| `__lt__` 等比较 | `<` `<=` `>` `>=` | 想用 `sorted` 排序就实现；可用 `functools.total_ordering` 自动补全 |
| `__len__` / `__getitem__` / `__setitem__` | `len()` / `obj[i]` / `obj[i]=v` | 容器协议；实现 `__getitem__` 的对象自动可迭代 |
| `__iter__` / `__next__` | `for` | 迭代协议（见 §4） |
| `__call__` | `obj()` | 让实例可被当成函数调用 |
| `__enter__` / `__exit__` | `with` | 上下文管理器；`__exit__` 返回 True 吞异常 |
| `__getattr__` / `__setattr__` | 属性不存在时 / 赋值 | 动态属性（小心递归陷阱，用 `object.__getattr__`） |
| `__slots__` | 类定义 | 声明固定属性集合，**省内存**（去掉 `__dict__`/`__weakref__`）、防手滑新增属性 |

### 5.3 三个「类级别」装饰器对比

| 装饰器 | 第一参数 | 场景 |
| --- | --- | --- |
| `@staticmethod` | 无 | 纯工具函数，不依赖实例/类 |
| `@classmethod` | `cls` | 工厂方法、访问类属性、**子类继承时用子类** |
| `@property` | `self` | 把方法变属性；配 `@x.setter` 做校验 |

### 5.4 继承与 MRO（方法解析顺序）

多继承时方法查找按 **MRO（C3 线性化）** 顺序：`cls.__mro__` 可查。菱形继承（A→B,C→D）下，`D.__mro__` 保证**每个基类只出现一次**且顺序稳定，`super()` 沿 MRO 向后传播——这是 Python 多继承不混乱的原因。


In [ ]:
class Point:
    _count = 0
    def __init__(self, x, y):
        self.x, self.y = x, y
        Point._count += 1
    def __repr__(self):
        return f'Point({self.x}, {self.y})'
    def __eq__(self, other):
        return (self.x, self.y) == (other.x, other.y)
    def __hash__(self):
        return hash((self.x, self.y))
    def __len__(self):
        return 2
    def __call__(self):
        return self.x + self.y
    @classmethod
    def from_tuple(cls, t):
        return cls(*t)
    @staticmethod
    def distance(p, q):
        return ((p.x - q.x) ** 2 + (p.y - q.y) ** 2) ** 0.5
    @property
    def norm(self):
        return (self.x ** 2 + self.y ** 2) ** 0.5

p1 = Point(1, 2)
p2 = Point(1, 2)
p3 = Point(3, 4)
print('repr   :', p1)
print('eq/hash:', p1 == p2, hash(p1) == hash(p2), '| set 去重:', {p1, p2, p3})
print('len/call:', len(p1), p1())
print('classmethod:', Point.from_tuple((5, 6)))
print('staticmethod:', Point.distance(p1, p3))
print('property:', p1.norm)

# 上下文管理器
class Managed:
    def __enter__(self):
        print('enter')
        return self
    def __exit__(self, *exc):
        print('exit', exc[0])
        return False   # False=不吞异常

with Managed() as m:
    print('inside')

# MRO 演示
class A:
    def f(self): return 'A'
class B(A):
    def f(self): return 'B'
class C(A):
    def f(self): return 'C'
class D(B, C):
    pass

print('D.f():', D().f(), '| MRO:', [k.__name__ for k in D.__mro__])

## 6. GIL 与并发选型

### 6.1 GIL 是什么、为什么存在

**GIL（Global Interpreter Lock，全局解释器锁）**是 CPython 的一个互斥锁：**同一时刻只有一个线程能执行 Python 字节码**。

为什么存在？——**引用计数不是线程安全的**。CPython 用引用计数管理内存，多线程并发增减引用计数会破坏计数（少释放=泄漏，多释放=悬垂）。给每个对象加锁开销太大，干脆「全局一把锁，一次只放一个线程执行字节码」，实现简单、单线程性能好（历史原因 + C 扩展兼容）。

**代价**：多线程**无法并行**执行纯 Python 的 CPU 密集计算（并行度受限）。注意区分——**并发（concurrency：处理多个任务）≠ 并行（parallelism：同时执行多任务）**，GIL 杀掉的是后者。

**何时释放 GIL**：

- 线程**阻塞在 I/O**（读文件/网络/`sleep`）时主动释放；
- C 扩展（NumPy 部分操作、`hashlib`、`re` 的部分路径）执行时释放，所以「NumPy 计算不吃 GIL」；
- Python 字节码：每执行一段就强制切换（旧版 100 条字节码、3.10+ 改为超时切换），保证多线程公平。

### 6.2 选型建议

| 任务类型 | 推荐 | 原因 |
| --- | --- | --- |
| I/O 密集（网络/磁盘/爬虫/API） | `threading` 或 `asyncio` | 等 I/O 时释放 GIL，多线程可并发重叠等待 |
| CPU 密集（纯 Python 计算） | `multiprocessing` | 每进程独立解释器与 GIL，真多核并行 |
| 海量轻量 I/O（数千并发连接） | `asyncio` 协程 | 单线程事件循环，无线程切换成本，内存开销最小 |

**asyncio 补充**：

- `async def` + `await`，事件循环（`asyncio.run`）驱动，`await` 点让出控制权；
- 优点：百万级并发连接、无锁竞态（单线程）、调度可控；
- 缺点：代码里不能有阻塞调用；CPU 密集任务会卡死事件循环（需丢给线程池/进程池）。

### 6.3 垃圾回收补充

CPython 内存管理**以引用计数为主**（对象引用数归零立即析构），**标记-清除 + 分代回收为辅**（解决循环引用：a 引用 b、b 引用 a 时引用计数永不为零）。`gc` 模块可手动触发，`weakref` 可建立不增加计数的引用（破环）。

> ⚠️ **易错点**：别答「Python 不支持多线程」。多线程对 I/O 完全有效；受限的只是「多线程并行跑纯 Python 的 CPU 计算」。也别答「Python 没有内存泄漏」——长期 GC 不到的大对象、闭包持有大引用、全局缓存都会造成事实泄漏。


In [ ]:
# 演示：GIL 下 CPU 密集多线程无加速；I/O 场景不在本环境演示
import threading, time

def cpu_work():
    s = 0
    for i in range(3 * 10**6):
        s += i
    return s

t0 = time.perf_counter(); r1 = cpu_work(); t1 = time.perf_counter()
threads = [threading.Thread(target=cpu_work) for _ in range(4)]
t2 = time.perf_counter()
for th in threads: th.start()
for th in threads: th.join()
t3 = time.perf_counter()
print(f'单线程: {t1-t0:.3f}s | 4线程(GIL串行): {t3-t2:.3f}s')
print('<- 纯 Python CPU 任务：多线程几乎无加速（GIL 串行）')
print('   CPU 密集请用 multiprocessing（每进程独立 GIL，真并行）')

# asyncio 概念演示：并发等待 I/O（用 asyncio.sleep 模拟）
import asyncio

async def fetch(i):
    await asyncio.sleep(0.05)
    return i

async def main():
    return await asyncio.gather(*(fetch(i) for i in range(8)))

t0 = time.perf_counter()
r = await main()  # Jupyter 内核已在事件循环里，用顶层 await 而非 asyncio.run()
t1 = time.perf_counter()
print(f'asyncio 并发 8 个 50ms 模拟I/O 总耗时: {t1-t0:.3f}s (≈50ms，并发重叠), 结果: {r}')

## 7. 性能优化速记

1. **内置函数/库优先**：`sum` / `sorted` / `max` / `min` / `itertools` 是 C 实现，远快于手写 Python 循环；
2. **推导式 > for + append**（少一次 `list.append` 属性查找）；
3. **局部变量 > 全局变量**：字节码层面 `LOAD_FAST`（数组索引）比 `LOAD_GLOBAL`（dict 查找）快；把常用全局（如 `math.sin`）绑定到局部再循环；
4. **`join` 拼接字符串**：`''.join(parts)` O(n)，`s += x` 循环最坏 O(n²)（虽然 CPython 有内联优化，但依赖实现细节）；
5. **`set`/`dict` 查重 O(1)**：替代 `list` 线性查 O(n)，大数据量差几个数量级；
6. **切片/视图**：`itertools.islice`、`numpy` 视图避免拷贝；
7. **`__slots__` 省内存**：大量小对象时用 `__slots__` 去掉 `__dict__`（每个对象省约 100+ 字节）；
8. **分析优先**：先用 `timeit`/`cProfile` 定位瓶颈再优化，别凭感觉。

> **复杂度的坑**：能用 O(1) 查重就别用 O(n) 线性扫——下面实验用 `n=3000` 在你机器上也能清楚看到数量级差距（若 n 更大差距可达百倍千倍）。


In [ ]:
import time

# 1) 内置 vs 手写循环
data = list(range(10**6))
t0 = time.perf_counter(); r1 = sum(data); t1 = time.perf_counter()
s = 0
for x in data: s += x
t2 = time.perf_counter()
print(f'内置 sum: {t1-t0:.4f}s, 手写循环: {t2-t1:.4f}s (结果一致: {r1==s})')

# 2) 推导式 vs 循环 append
t0 = time.perf_counter(); a = [x*x for x in range(10**6)]; t1 = time.perf_counter()
b = []
for x in range(10**6): b.append(x*x)
t2 = time.perf_counter()
print(f'推导式: {t1-t0:.4f}s, append 循环: {t2-t1:.4f}s')

# 4) join vs 字符串 +
t0 = time.perf_counter(); s1 = ''.join(str(x) for x in range(10**5)); t1 = time.perf_counter()
s2 = ''
for x in range(10**5): s2 += str(x)
t2 = time.perf_counter()
print(f'join: {t1-t0:.4f}s, += 循环: {t2-t1:.4f}s')

# 5) set 查重 vs list 线性查
n = 3000
pool = set(range(n)); lst = list(range(n))
qs = list(range(n))
t0 = time.perf_counter(); h1 = sum(q in pool for q in qs); t1 = time.perf_counter()
t2 = time.perf_counter(); h2 = sum(q in lst for q in qs); t3 = time.perf_counter()
print(f'set 查重(哈希 O(1)): {t1-t0:.4f}s 命中{h1}')
print(f'list 查重(线性 O(n)): {t3-t2:.4f}s 命中{h2}')
print(f'耗时比 ~{max((t3-t2)/(t1-t0+1e-12),1):.0f}x (n 越大差距越大)')

## 8. 自测清单（面试追问）

- [ ] `a = b` / `copy.copy` / `copy.deepcopy` 三者区别？嵌套列表改内层各是什么结果？
- [ ] 为什么 `tuple` 可作 dict 键而 `list` 不行？「元组不可变」一定意味着内容不可变吗？
- [ ] `is` 与 `==` 的区别？为什么小整数 `is` 为 True、大整数为 False？
- [ ] 函数默认参数用可变对象有什么坑？怎么修？
- [ ] 闭包循环陷阱是什么？怎么修？（默认参数 / `functools.partial`）
- [ ] 装饰器为什么用 `functools.wraps`？`@property` / `@staticmethod` / `@classmethod` 区别？多个装饰器执行顺序？
- [ ] 生成器与迭代器的关系？`yield` 挂起/恢复的原理？`yield from` 和 `send` 是什么？
- [ ] 定义了 `__eq__` 后为什么对象可能不可哈希？`__slots__` 有什么用？
- [ ] GIL 是什么？为什么存在？I/O 密集 vs CPU 密集分别用什么？为什么？
- [ ] asyncio 与 threading 的适用场景？事件循环里跑 CPU 密集会发生什么？
- [ ] `str +=` 与 `list +=` 在内存上有什么本质区别？
- [ ] 手写：一行找出列表中出现次数最多的元素（`Counter` 或 `max(dict, key=dict.get)`）。
- [ ] 手写：用生成器实现 `range` 的 `__next__`；实现一个带 `__enter__`/`__exit__` 的计时上下文。

> ✅ 全部能回答 + 能默写代码，本课通过。